# Problematica
Se trabaja con el conjunto de datos oficial "tb_patentes_aguas_december_2025_data.csv", que consolida las
patentes por no uso de derechos de aprovechamiento de aguas en Chile (46.467 registros, 21 variables; Tesorería
General de la República, 2025). <br>
## Pregunta de análisis: ¿cómo se distribuyen la morosidad y el saldo impago de estas
patentes según Tesorería Regional, tipo de titular (persona natural o jurídica) y año de emisión? 
<br>
## Objetivo metodológico: 
que el análisis sea trazable y reproducible mediante Git/GitHub y Jupyter Notebooks.


### Herramientas del ecosistema científico

| Pieza | Por que se usa | 
|---|---|
| Entorno virtual (`.venv`) | Aislar las versiones de las librerías del proyecto. | 
| JupyterLab + `ipykernel` | Ejecutar el cuaderno con el intérprete del proyecto. | 
| `requirements.txt` | Declarar las dependencias exactas. | 
| Git | Registrar la historia del trabajo y su autoría. | 
| GitHub | Colaborar y publicar la evidencia evaluada. | 
| `pathlib` | Rutas relativas independientes del sistema operativo. | 

En esta fase se usan sobre todo módulos de la biblioteca estándar: el cuaderno construye y
verifica el entorno, todavía no analiza datos. `numpy` y `pandas` aparecen únicamente para
comprobar que están disponibles y para escribir las tablas de documentación.

In [ ]:
import sys                        # intérprete en uso: es lo que se verifica más abajo
import json                       # exportación de metadatos legibles
import platform                   # sistema operativo, para dejarlo en la bitácora
import subprocess                 # consultas a Git desde el cuaderno
import shutil                     # localizar ejecutables (git) sin suponer que existen
import importlib                  # importar el módulo que este cuaderno va a escribir
import importlib.util             # comprobar si un paquete está instalado
from pathlib import Path          # manejo de rutas independiente del sistema operativo
from datetime import date

# Verificación del entorno: instala en el intérprete del kernel lo que falte.
print("Intérprete del kernel:", sys.executable)

REQUERIDOS = {"numpy": "numpy", "pandas": "pandas"}   # módulo: paquete pip
faltantes = [pkg for mod, pkg in REQUERIDOS.items()
             if importlib.util.find_spec(mod) is None]

if faltantes:
    print("Instalando:", ", ".join(faltantes))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *faltantes])
    importlib.invalidate_caches()

import numpy as np                # presente para verificar el entorno del proyecto
import pandas as pd               # tablas de documentación de la fase

print("numpy", np.__version__, "· pandas", pd.__version__)

# Reproducibilidad: la misma semilla que usa el cuaderno de la Fase 2.
SEMILLA = 42
np.random.seed(SEMILLA)

print("Cuaderno de la Fase 1 ·", date.today().isoformat())

## 1. Definición del problema

Deja el problema escrito como una **estructura de datos**

In [ ]:
PROYECTO = {
    "titulo": "consolidacion de patentes por no uso de derechos de aprovechamiento de aguas",
    "grupo": "Grupo 9",
    "asignatura": "MCDI500 · Programación para la Ciencia de Datos",
    "integrantes": ["Christian Abarca Araya", "Vicente Agosin Orellana", "Patricia Campos Tapia","Daniel Valdivia Tapia"],
    "problematica": (
        

        
        "Análisis de la morosidad en las patentes por no uso de derechos de aprovechamiento de aguas en Chile, a partir del "
        "conjunto de datos oficial publicado en el Portal de Datos Abiertos del Gobierno de Chile. El objetivo es describir cómo "
        "se distribuyen la morosidad y el saldo impago según Tesorería Regional, tipo de titular (persona natural o jurídica) y "
        "año de emisión, con un flujo de trabajo trazable y reproducible. "
    ),
    "objetivo_general": (
        "Construir un flujo reproducible en Python, versionado en Git/GitHub y ejecutado en JupyterLab, que procese el registro de patentes por no uso de aguas (46.467 cobros, 2007–2025)" 
         "y cuantifique cómo se distribuye la morosidad según tesorería regional, tipo de titular y año de la patente." 
 
    ),
    "objetivos_especificos": [
        "Preparar un conjunto confiable y sin datos personales (F2).",
        "Construir el pipeline de obtención, limpieza y transformación de datos (F2).",
        "Implementar el núcleo algorítmico con programación estructurada y POO (F3).",
        "Comunicar los hallazgos mediante visualización y un informe técnico (F4).",
    ],
    "preguntas": [
        "En qué tesorería regional se concentran las mayores morosidades por no aprovechamiento de aguas",
        "Qué tipo de titular concentra mayor morosidad por no aprovechamiento de aguas",
    ],
    "criterios_exito": [
        "El repositorio se clona y el cuaderno corre completo sin intervención manual.",
        "Cada decisión de preprocesamiento queda justificada con cifras, no con adjetivos.",
        "Los cuatro integrantes tienen commits propios en el historial.",
    ],
    "alcance": {
        "incluye": ["Definición del problema", "Entorno reproducible", "Selección del conjunto"],
        "excluye": ["Limpieza e imputación", "Modelado predictivo", "Despliegue"],
    },
    "limitaciones": [
        "Los datos se toman tal como los publica la Tesorería; no se verifica su exactitud contra otras fuentes.",
        "691 registros (1,49 %) no tienen provincia, año ni correlativo de patente, y 34 (0,07 %) no tienen tesorería.",
        "Es un corte único a diciembre de 2025: describe la morosidad en ese momento, no su evolución entre publicaciones.",
        "El archivo contiene el RUT de los titulares: no se sube al repositorio ni se muestra en los resultados.",
    ],
}

In [ ]:
def presentar_proyecto(config):
    """
    Imprime la definicion del proyecto de forma legible.

    Parametros
    ----------
    config : dict
        Diccionario de configuracion del proyecto.

    Retorna
    -------
    None

    Lanza
    -----
    KeyError
        Si falta alguna de las claves obligatorias.
    """
    obligatorias = ("titulo", "problematica", "objetivo_general", "objetivos_especificos")
    faltantes = [c for c in obligatorias if c not in config]
    if faltantes:
        # Se falla temprano y con un mensaje que dice QUE falta, no solo que algo falló.
        raise KeyError(f"Faltan claves obligatorias en la configuracion: {faltantes}")

    print(config["titulo"].upper())
    print("=" * 60)
    print(f"{config['asignatura']} · {config['grupo']}")
    print("\nProblemática")
    print(config["problematica"])
    print("\nObjetivo general")
    print(config["objetivo_general"])
    print("\nObjetivos específicos")
    for i, obj in enumerate(config["objetivos_especificos"], start=1):
        print(f"  {i}. {obj}")
    print("\nPreguntas que orientan el trabajo")
    for pregunta in config["preguntas"]:
        print(f"  · {pregunta}")


presentar_proyecto(PROYECTO)

In [ ]:
print("Alcance de la Fase 1")
print("  Incluye:", ", ".join(PROYECTO["alcance"]["incluye"]))
print("  Excluye:", ", ".join(PROYECTO["alcance"]["excluye"]))

print("\nLimitaciones declaradas")
for lim in PROYECTO["limitaciones"]:
    print(f"  · {lim}")

## 2. Verificación del entorno

**Qué hace este paso.** Comprueba, con evidencia y no por confianza, que el cuaderno se
está ejecutando con el intérprete del proyecto y que las librerías declaradas están
disponibles en él.

In [ ]:
DEPENDENCIAS = ["numpy", "pandas", "matplotlib", "sklearn"]

# Correspondencia entre el nombre de importación y el nombre del paquete en PyPI:
# se instala scikit-learn, pero se importa sklearn.
NOMBRE_EN_PYPI = {"sklearn": "scikit-learn"}


def verificar_entorno(dependencias):
    """
    Comprueba interprete, entorno virtual, carpeta de trabajo y librerias.

    Parametros
    ----------
    dependencias : list of str
        Nombres de importacion de las librerias que el proyecto declara.

    Retorna
    -------
    dict
        Resultado de cada comprobacion, para dejarlo en la bitacora de la fase.
    """
    reporte = {}

    # 1. Intérprete que ejecuta ESTE cuaderno. Si la ruta no contiene .venv,
    #    el kernel no es el del proyecto: Kernel -> Change Kernel.
    ejecutable = Path(sys.executable)
    en_venv = ".venv" in ejecutable.parts or sys.prefix != sys.base_prefix
    reporte["interprete"] = str(ejecutable)
    reporte["entorno_virtual"] = bool(en_venv)
    print("Intérprete       :", ejecutable)
    print("Entorno virtual  :", "[OK] activo" if en_venv else "[AVISO] parece el Python del sistema")

    # 2. Carpeta de trabajo: las rutas relativas se resuelven desde aquí,
    #    no desde donde está guardado el archivo .ipynb.
    reporte["carpeta_trabajo"] = str(Path.cwd())
    print("Carpeta de trabajo:", Path.cwd())
    print("Sistema          :", platform.system(), platform.release())
    print("Python           :", sys.version.split()[0])

    # 3. Librerías del proyecto, con su versión.
    print("\nLibrerías declaradas")
    versiones = {}
    for nombre in dependencias:
        try:
            modulo = importlib.import_module(nombre)
            version = getattr(modulo, "__version__", "sin atributo __version__")
            versiones[nombre] = version
            print(f"  [OK]    {nombre:12} {version}")
        except ImportError:
            # No se interrumpe el cuaderno: se informa qué instalar y con qué nombre.
            versiones[nombre] = None
            paquete = NOMBRE_EN_PYPI.get(nombre, nombre)
            print(f"  [FALTA] {nombre:12} instale con: python -m pip install {paquete}")
    reporte["versiones"] = versiones
    return reporte


ENTORNO = verificar_entorno(DEPENDENCIAS)

## 3. Estructura del repositorio y artefactos

**Qué hace este paso.** Crea las carpetas del proyecto y escribe desde el cuaderno los dos
archivos que hacen reproducible el entorno: `.gitignore` y `requirements.txt`.

**Por qué desde el cuaderno.** Porque así quedan derivados de la configuración declarada en
la sección 1 y no de lo que alguien recordó escribir a mano. Es la misma lógica de la Fase
2: el artefacto se genera, se verifica y queda trazado.

In [ ]:
def localizar_raiz(inicio=None):
    """
    Devuelve la raíz del proyecto.

    Jupyter abre el kernel en la carpeta donde está guardado el cuaderno, no en la
    carpeta desde la que se lanzó JupyterLab. Como cada cuaderno vive en la carpeta
    de su fase (F1, F2, F3 o F4), si la carpeta de trabajo es una de ellas la raíz
    es la carpeta superior. Si el cuaderno se ejecuta desde la raíz, se usa tal cual.
    """
    inicio = Path.cwd() if inicio is None else Path(inicio)
    return inicio.parent if inicio.name in {"F1", "F2", "F3", "F4"} else inicio


RAIZ = localizar_raiz()

# Rutas de datos COMUNES a todas las fases: son exactamente las que usa el
# cuaderno de la Fase 2, de modo que un mismo archivo sirva a los dos.
DIR_CRUDO = RAIZ / "data" / "raw"          # datos originales: nunca se modifican
DIR_PROCESADO = RAIZ / "data" / "processed"  # salida del pipeline de la Fase 2
DIR_DOCS = RAIZ / "docs"                   # diccionario, fichas y metadatos
DIR_SRC = RAIZ / "src"                     # módulos propios reutilizables

# Una carpeta por fase para cuadernos e informes.
DIR_FASES = [RAIZ / f"F{n}" for n in (1, 2, 3, 4)]

for carpeta in [DIR_CRUDO, DIR_PROCESADO, DIR_DOCS, DIR_SRC, *DIR_FASES]:
    # parents=True crea las intermedias; exist_ok=True evita el error si ya existen,
    # que es lo que permite volver a ejecutar el cuaderno completo sin romperlo.
    carpeta.mkdir(parents=True, exist_ok=True)

print("Estructura creada bajo:", RAIZ.resolve())
# Se omiten las carpetas ocultas (.git, .venv, .ipynb_checkpoints) y las cachés.
for ruta in sorted(p.relative_to(RAIZ) for p in RAIZ.rglob("*") if p.is_dir()):
    if any(parte.startswith(".") or parte == "__pycache__" for parte in ruta.parts):
        continue
    print("  ", ruta.as_posix() + "/")

### 3.1 El archivo `.gitignore`

Quedan fuera del repositorio tres cosas: lo que se regenera (el entorno virtual, las
cachés), lo que es local de cada equipo (archivos del sistema operativo) y lo que es
secreto (credenciales). Se escribe **antes** del primer `git add .`: después ya es tarde,
porque el archivo quedó en el historial.

In [ ]:
CONTENIDO_GITIGNORE = """# Entorno virtual: se reconstruye con requirements.txt
.venv/

# Caché de Python
__pycache__/
*.pyc

# Jupyter
.ipynb_checkpoints/

# Windows
Thumbs.db
desktop.ini

# macOS
.DS_Store
.AppleDouble

# Variables de entorno y secretos
.env

# Datos originales: contienen el RUT de los titulares, no se suben
data/raw/

# Archivos de bloqueo de LibreOffice y Excel
.~lock.*
~$*

# Datos pesados y modelos
*.ckpt
*.pth
"""

ARCHIVO_GITIGNORE = RAIZ / ".gitignore"
ARCHIVO_GITIGNORE.write_text(CONTENIDO_GITIGNORE, encoding="utf-8")

# Verificación de ida y vuelta: se relee lo escrito antes de darlo por hecho.
lineas = [l for l in ARCHIVO_GITIGNORE.read_text(encoding="utf-8").splitlines()
          if l and not l.startswith("#")]
assert ".venv/" in lineas, "El entorno virtual debe estar ignorado."
assert "data/raw/" in lineas, "Los datos originales (con RUT) deben estar ignorados."
print(f"{ARCHIVO_GITIGNORE} escrito con {len(lineas)} reglas activas.")

### 3.2 El archivo `requirements.txt`

Declarar las dependencias es lo que separa «funciona en mi computador» de un entorno
reproducible. Se escriben con la versión exacta observada en el entorno, no con la última
que exista en el momento de instalar.

In [ ]:
def escribir_requirements(ruta, versiones):
    """
    Escribe requirements.txt a partir de las versiones observadas en el entorno.

    Parametros
    ----------
    ruta : Path
        Archivo de salida.
    versiones : dict
        Nombre de importacion -> version detectada (o None si falta).

    Retorna
    -------
    list of str
        Lineas escritas, para verificarlas.

    Lanza
    -----
    ValueError
        Si ninguna dependencia pudo detectarse.
    """
    lineas = []
    for nombre, version in versiones.items():
        if version is None:
            continue  # una librería ausente no se declara: primero se instala
        paquete = NOMBRE_EN_PYPI.get(nombre, nombre)
        lineas.append(f"{paquete}=={version}")

    if not lineas:
        raise ValueError("No se detectó ninguna dependencia instalada.")

    # Se agregan las herramientas del flujo de trabajo, que no se importan
    # desde el código pero sí forman parte del entorno reproducible.
    lineas += ["jupyterlab", "notebook", "ipykernel", "nbstripout", "jupytext", "jupyterlab-git"]
    ruta.write_text("\n".join(sorted(lineas)) + "\n", encoding="utf-8")
    return sorted(lineas)


ARCHIVO_REQUIREMENTS = RAIZ / "requirements.txt"
requisitos = escribir_requirements(ARCHIVO_REQUIREMENTS, ENTORNO["versiones"])
print(ARCHIVO_REQUIREMENTS, "\n")
print("\n".join(requisitos))

## 4. Módulos: del cuaderno a `src/`

**Qué hace este paso.** Escribe un módulo `src/utilidades.py` y lo importa de vuelta.

**Por qué.** La Unidad 1 pide funciones, excepciones **y módulos**. Un cuaderno documenta y
comunica; un módulo es lo que permite que la misma función se use en las cuatro fases sin
copiarla. Aquí se mueve una función pequeña pero real: la que normaliza los nombres de las
columnas, que las fases siguientes usarán apenas carguen datos.

In [ ]:
CODIGO_MODULO = """\
\"\"\"Utilidades compartidas del proyecto MCDI500 - Grupo 9.

Este modulo lo genera el cuaderno de la Fase 1 y lo importan las fases siguientes.
\"\"\"

import re


class ProyectoError(Exception):
    \"\"\"Error propio del proyecto: permite distinguirlo de los de las librerias.\"\"\"


def normalizar_nombre(texto):
    \"\"\"Convierte un nombre de columna a minusculas, sin espacios ni signos.

    Ejemplo
    -------
    >>> normalizar_nombre('Residence Type ')
    'residence_type'
    \"\"\"
    if not isinstance(texto, str):
        raise ProyectoError(f\"Se esperaba texto y se recibio {type(texto).__name__}.\")
    limpio = texto.strip().lower()
    limpio = re.sub(r\"[^a-z0-9]+\", \"_\", limpio)   # todo lo que no sea letra o digito -> _
    return limpio.strip(\"_\")


def normalizar_columnas(nombres):
    \"\"\"Aplica normalizar_nombre a una lista y verifica que no se produzcan duplicados.\"\"\"
    normalizados = [normalizar_nombre(n) for n in nombres]
    if len(set(normalizados)) != len(normalizados):
        raise ProyectoError(\"La normalizacion produjo nombres duplicados.\")
    return normalizados
"""

ARCHIVO_MODULO = DIR_SRC / "utilidades.py"
ARCHIVO_MODULO.write_text(CODIGO_MODULO, encoding="utf-8")
print("Módulo escrito en:", ARCHIVO_MODULO)

Para importarlo hay que agregar `src/` a la lista de rutas donde Python busca módulos.
`invalidate_caches()` es necesario porque el archivo se acaba de crear: sin esa llamada,
Python puede no ver un módulo escrito después de arrancar el intérprete.

In [ ]:
if str(DIR_SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(DIR_SRC.resolve()))

importlib.invalidate_caches()
import utilidades                      # el módulo recién escrito
importlib.reload(utilidades)           # por si el cuaderno se vuelve a ejecutar

print("Importado:", utilidades.__name__, "desde", Path(utilidades.__file__).as_posix())
print(utilidades.normalizar_nombre("TIPO_PERSONA"))
print(utilidades.normalizar_columnas(["TESORERIA_REGIONAL", "COMUNA", "TIPO_PERSONA"]))

## 5. Selección y ficha del conjunto de datos

**Qué hace este paso.** Documenta la procedencia del conjunto y la evalúa contra los
criterios de selección del curso.

**Por qué en la Fase 1.** Porque la elección del conjunto condiciona todo lo que viene
después. Un conjunto ya limpio convierte la sección de preprocesamiento de la Fase 2 en un
trámite y hace perder el criterio de mayor peso de la Sumativa 1.

### 5.1 Procedencia

| Campo | Valor |
| --- | --- |
| Título | consolidacion de patentes por no uso de derechos de aprovechamiento de aguas|
| Autor | Tesoreía General de La República |
| Plataforma | Datos.gob.cl|
| Enlace | https://datos.gob.cl/dataset/tb-patentes-aguas/resource/4d1d615f-5935-4760-b399-1dd76b281715 |
| Estructura esperada | 46.468 filas × 21 columnas originales· 34 nulos en `tesoreria_regional` |
| Unidad de observación | Pago patentes de agua |
| Licencia | Abierta, declarada en la plataforma |

**Referencia en APA 7 para el informe:**
Tesorería General de la República. (2025). Reporte consolidado de recaudación y cartera morosa de patentes de aguas (diciembre 2025) [Conjunto de datos]. Ministerio de Hacienda, Gobierno de Chile. https://datos.gob.cl/dataset/tb-patentes-aguas/resource/4d1d615f-5935-4760-b399-1dd76b281715

Documentar la procedencia no es un trámite: sin ella nadie puede verificar sobre qué
versión del archivo se trabajó.

In [ ]:
FICHA = {
    "titulo": "Consolidación de patentes por no uso de derechos de aprovechamiento de aguas",
    "autor": "Tesorería General de la República",
    "plataforma": "Portal de Datos Abiertos del Gobierno de Chile",
    "url": "https://datos.gob.cl/dataset/tb-patentes-aguas",
    "licencia_abierta": True,
    "unidad_observacion": "Una patente",
    "filas": 46467,
    "columnas": 21,
    "tamano_mb": 12.6,
    # Conteo de variables por rol analítico: debe coincidir con el diccionario de la sección 5.3.
    "roles": {
        "continua": 6,
        "discreta": 2,
        "binaria": 1,
        "nominal": 1,
        "ordinal": 0,
        "fecha": 2,
        "alta_cardinalidad": 7,
        "identificador": 2,
    },
    "pct_nulos_max_variable": 1.49,   # PROVINCIA, N_DE_CORRELATIVO_O_PATENTE y ANIO_PATENTE (691 registros)
    "pct_nulos_min_no_cero": 0.07,    # TESORERIA y TESORERIA_REGIONAL (34 registros)
}

### 5.2 Evaluación contra los criterios del curso

La función compara la ficha con los mínimos definidos para el proyecto y devuelve una
tabla con el veredicto por criterio. No decide por el grupo: expone lo que falta.

In [ ]:
# Requisitos del curso. Los OBLIGATORIOS descalifican el conjunto si no se cumplen;
# los AVISOS no lo descalifican, pero lo que falte se declara en el informe como
# restricción del conjunto. Son los mismos que aplica validar_dataset.py.
OBLIGATORIOS = {
    "filas": ("Filas", 2000, "Permite agrupar por categoría sin grupos vacíos"),
    "columnas": ("Columnas", 12, "Asegura variedad de roles analíticos"),
    "roles": ("Roles analíticos distintos", 3, "El conjunto debe ser multivariado"),
    "numerica": ("Variables numéricas", 1, "Escalamiento y valores atípicos"),
    "categorica": ("Variables categóricas", 1, "Codificación de categorías"),
}
AVISOS = {
    "fecha": ("Fecha", 1, "Parseo y derivación de variables temporales"),
    "alta_cardinalidad": ("Texto o alta cardinalidad", 1, "Normalización y agrupación"),
}

# Roles que cuentan para cada grupo. El identificador no es un rol analítico:
# no se analiza, solo distingue filas.
NUMERICOS = ("continua", "discreta")
CATEGORICOS = ("binaria", "nominal", "ordinal")


def evaluar_criterios(ficha, obligatorios=OBLIGATORIOS, avisos=AVISOS):
    """
    Compara una ficha de conjunto de datos con los requisitos del curso.

    Parametros
    ----------
    ficha : dict
        Ficha del conjunto, con las claves 'filas', 'columnas' y 'roles'.
    obligatorios, avisos : dict
        Criterio -> (etiqueta, minimo, motivo).

    Retorna
    -------
    pd.DataFrame
        Una fila por criterio, con su tipo, el valor observado y el veredicto.

    Lanza
    -----
    KeyError
        Si la ficha no declara los roles analiticos.
    """
    if "roles" not in ficha:
        raise KeyError("La ficha debe declarar el conteo de variables por rol analítico.")

    roles = ficha["roles"]
    observado = {
        "filas": ficha.get("filas", 0),
        "columnas": ficha.get("columnas", 0),
        # Cuántos roles distintos aparecen al menos una vez, sin contar el identificador.
        "roles": sum(1 for rol, n in roles.items() if n > 0 and rol != "identificador"),
        "numerica": sum(roles.get(r, 0) for r in NUMERICOS),
        "categorica": sum(roles.get(r, 0) for r in CATEGORICOS),
        "fecha": roles.get("fecha", 0),
        "alta_cardinalidad": roles.get("alta_cardinalidad", 0),
    }

    filas = []
    for tipo, criterios in (("obligatorio", obligatorios), ("aviso", avisos)):
        for clave, (etiqueta, minimo, motivo) in criterios.items():
            valor = observado[clave]
            # Un aviso incumplido no dice NO: dice que hay algo que declarar.
            falla = "NO" if tipo == "obligatorio" else "aviso"
            filas.append({
                "criterio": etiqueta,
                "tipo": tipo,
                "mínimo": minimo,
                "observado": valor,
                "cumple": "sí" if valor >= minimo else falla,
                "por qué se pide": motivo,
            })

    # Los faltantes se informan como aviso: un conjunto sin ellos se puede usar,
    # pero deja con poca evidencia la sección de preprocesamiento de la Fase 2.
    pct = ficha.get("pct_nulos_min_no_cero", 0.0)
    filas.append({
        "criterio": "Alguna variable con ≥1% de nulos", "tipo": "aviso",
        "mínimo": 1.0, "observado": pct,
        "cumple": "sí" if pct >= 1.0 else "aviso",
        "por qué se pide": "Un conjunto perfecto no tiene preprocesamiento que justificar",
    })
    pct_max = ficha.get("pct_nulos_max_variable", 0.0)
    filas.append({
        "criterio": "Ninguna variable sobre 60% de nulos", "tipo": "aviso",
        "mínimo": 60.0, "observado": pct_max,
        "cumple": "sí" if pct_max <= 60.0 else "aviso",
        "por qué se pide": "Por encima de eso la variable no es recuperable",
    })
    tabla = pd.DataFrame(filas)
    # Formato: los mínimos y los valores observados se muestran sin decimales
    # innecesarios; la comparación ya se hizo sobre los números.
    for col in ("mínimo", "observado"):
        tabla[col] = tabla[col].map(lambda v: f"{v:g}")
    return tabla


evaluacion = evaluar_criterios(FICHA)
evaluacion

In [ ]:
incumplidos = evaluacion.loc[evaluacion["cumple"] == "NO", "criterio"].tolist()
avisos = evaluacion.loc[evaluacion["cumple"] == "aviso", "criterio"].tolist()
print(f"Criterios evaluados : {len(evaluacion)}")
print(f"Obligatorios sin cumplir: {len(incumplidos)}")
print(f"Avisos              : {len(avisos)}")
if incumplidos:
    print("\nNo cumple (descalifica el conjunto):")
    for c in incumplidos:
        print("  ·", c)
if avisos:
    print("\nAvisos (no descalifican; se declaran en el informe):")
    for c in avisos:
        print("  ·", c)

### 5.3 Diccionario de variables

El diccionario declara el **rol analítico** de cada variable. Es la decisión que determina
todo el preprocesamiento de la Fase 2: dos columnas `int64` pueden ser un identificador,
una variable discreta o una binaria, y cada una exige un tratamiento distinto.

In [ ]:
import pandas as pd

DICCIONARIO = [
    ("CUENTA", "identificador", "Identificador único de la cuenta"),
    ("RUT_ROL", "identificador", "RUT del titular, sin dígito verificador (dato personal: no se publica)"),
    ("TIPO_PERSONA", "binaria", "Tipo de persona registrado"),
    ("RUT_ROL_DV", "nominal", "Dígito verificador del RUT"),
    ("FORM_FOLIO", "continua", "Número de folio del formulario"),
    ("MONTO", "continua", "Monto asociado a la patente"),
    ("SALDO", "continua", "Saldo registrado de la patente"),
    ("PAGADO", "continua", "Monto pagado"),
    ("FECHA", "fecha", "Fecha del registro"),
    ("MOV_TIPO", "discreta", "Código del tipo de movimiento"),
    ("TESORERIA", "alta cardinalidad", "Tesorería asociada al registro"),
    ("TESORERIA_REGIONAL", "alta cardinalidad", "Tesorería regional correspondiente"),
    ("COMUNA", "alta cardinalidad", "Comuna asociada a la patente"),
    ("CONCESION", "alta cardinalidad", "Identificación de la concesión"),
    ("PROVINCIA", "alta cardinalidad", "Provincia asociada al registro"),
    ("OFICINA_CBR", "alta cardinalidad", "Oficina del Conservador de Bienes Raíces"),
    ("N_DE_CORRELATIVO_O_PATENTE", "continua", "Número correlativo o de patente"),
    ("ANIO_PATENTE", "discreta", "Año correspondiente a la patente"),
    ("UTM", "continua", "Valor registrado en UTM"),
    ("OFICINA", "alta cardinalidad", "Oficina asociada al registro"),
    ("FECHA_EJECUCION", "fecha", "Fecha de ejecución del proceso"),
]

diccionario = pd.DataFrame(DICCIONARIO, columns=["variable", "rol_analitico", "descripcion"])

# Verificación: el diccionario debe cubrir exactamente las columnas declaradas en la FICHA (sección 5.1).
assert len(diccionario) == FICHA["columnas"], "El diccionario no coincide con la ficha."
print(diccionario["rol_analitico"].value_counts().to_string())
diccionario

### 5.4 Comprobación del archivo (si ya está descargado)

Esta celda no descarga nada. Comprueba si el archivo está en `data/raw` y, en ese caso,
contrasta la estructura real con la declarada en la ficha. Si aún no lo descargan, informa
qué falta y el cuaderno sigue corriendo: la Fase 1 no depende del archivo.

In [ ]:
ARCHIVO = DIR_CRUDO / "tb_patentes_aguas_december_2025_data.csv"

if ARCHIVO.exists():
    df_crudo = pd.read_csv(ARCHIVO)
    print("Archivo encontrado :", ARCHIVO.as_posix())
    print("Dimensiones reales :", df_crudo.shape)
    print("Declarado en ficha :", (FICHA["filas"], FICHA["columnas"]))
    coincide = df_crudo.shape == (FICHA["filas"], FICHA["columnas"])
    print("Coincidencia       :", "[OK]" if coincide else "[REVISAR] ficha y archivo difieren")
    print("\nNulos por columna (solo las que tienen):")
    print(df_crudo.isna().sum().loc[lambda s: s > 0].to_string())
else:
    df_crudo = None
    print("[PENDIENTE] El archivo aún no está en", DIR_CRUDO.as_posix())
    print("Descárguelo desde:", FICHA["url"])
    print("y guárdelo como  :", ARCHIVO.as_posix())

## 6. Control de versiones

**Qué hace este paso.** Consulta el estado de Git desde el propio cuaderno: si está
instalado, si la identidad está configurada, si la carpeta es un repositorio y qué
*commits* tiene.

**Por qué se consulta y no se ejecuta.** Este cuaderno **no** hace `git init`, `add` ni
`commit`. Versionar es una decisión de quien trabaja, y automatizarla desde un cuaderno
produce historiales sin sentido. Lo que sí corresponde es dejar constancia verificable del
estado del repositorio en el momento de la entrega.

In [ ]:
def git(*argumentos):
    """
    Ejecuta un comando de Git y devuelve su salida como texto.

    Retorna
    -------
    (bool, str)
        Exito de la ejecucion y salida (o mensaje de error).
    """
    if shutil.which("git") is None:
        return False, "Git no está instalado o no está en el PATH."
    try:
        proceso = subprocess.run(["git", *argumentos], capture_output=True,
                                 text=True, timeout=20)
    except (OSError, subprocess.SubprocessError) as e:
        return False, f"No se pudo ejecutar Git: {e}"
    if proceso.returncode != 0:
        # stderr trae el motivo real; se devuelve tal cual para no ocultarlo.
        return False, proceso.stderr.strip() or "Git terminó con error."
    return True, proceso.stdout.strip()


ok, salida = git("--version")
print("Git                :", salida if ok else f"[AVISO] {salida}")

for clave in ("user.name", "user.email"):
    ok_cfg, valor = git("config", "--get", clave)
    estado = valor if ok_cfg and valor else "[FALTA] configúrelo con git config --global"
    print(f"{clave:19}:", estado)

In [ ]:
es_repo, _ = git("rev-parse", "--is-inside-work-tree")

if es_repo:
    ok_rama, rama = git("rev-parse", "--abbrev-ref", "HEAD")
    print("Rama actual        :", rama if ok_rama else "sin commits todavía")

    ok_log, log = git("log", "--oneline", "-5")
    print("\nÚltimos commits")
    print(log if ok_log and log else "  (todavía no hay commits)")

    # --porcelain da una salida estable, pensada para ser leída por un programa.
    ok_st, estado = git("status", "--porcelain")
    pendientes = [l for l in estado.splitlines() if l.strip()] if ok_st else []
    print(f"\nArchivos con cambios sin registrar: {len(pendientes)}")
    for linea in pendientes[:10]:
        print("  ", linea)

    ok_rem, remoto = git("remote", "-v")
    print("\nRemoto             :", remoto.splitlines()[0] if ok_rem and remoto
          else "[FALTA] agregue el remoto con git remote add origin URL")
else:
    print("[PENDIENTE] Esta carpeta todavía no es un repositorio Git.")
    print("Desde la terminal, en la raíz del proyecto:")
    print("  git init")
    print("  git add .gitignore requirements.txt README.md")
    print('  git commit -m "docs: estructura inicial del proyecto"')

## 7. Validación técnica de la fase

Verificar significa demostrar, con evidencia, que el resultado es correcto. La función
`validar_fase1` reúne las comprobaciones de la fase y usa `assert`: si una condición no se
cumple, el cuaderno se detiene con un error, dejando constancia de la falla.

Se comprueba:
1. **Estructura** — que existan las carpetas comprometidas.
2. **Artefactos** — que `.gitignore` y `requirements.txt` estén escritos y no vacíos.
3. **Definición** — que la configuración del proyecto tenga sus claves obligatorias.
4. **Coherencia** — que el diccionario cubra exactamente las columnas de la ficha.
5. **Módulo** — que `src/utilidades.py` sea importable.

In [ ]:
def validar_fase1(raiz, config, ficha, diccionario):
    """
    Comprueba que la Fase 1 dejo lo que la rubrica exige.
    Lanza AssertionError si alguna comprobacion falla.
    """
    print("VALIDACION DE LA FASE 1")
    print("-" * 45)

    # 1. Estructura de carpetas
    esperadas = ["data/raw", "data/processed", "docs", "src", "F1", "F2", "F3", "F4"]
    faltan = [c for c in esperadas if not (raiz / c).is_dir()]
    assert not faltan, f"Faltan carpetas: {faltan}"
    print(f"[OK] Estructura completa ({len(esperadas)} carpetas)")

    # 2. Artefactos del entorno reproducible
    for nombre in (".gitignore", "requirements.txt"):
        archivo = raiz / nombre
        assert archivo.exists(), f"Falta {nombre}."
        assert archivo.stat().st_size > 0, f"{nombre} está vacío."
        print(f"[OK] {nombre} presente ({archivo.stat().st_size} bytes)")

    # 3. Definición del problema
    for clave in ("titulo", "problematica", "objetivo_general", "objetivos_especificos"):
        assert config.get(clave), f"La configuración no declara '{clave}'."
    assert len(config["objetivos_especificos"]) >= 3, "Se esperan al menos tres objetivos."
    print("[OK] Definición del problema completa")

    # 4. Coherencia entre ficha y diccionario
    assert len(diccionario) == ficha["columnas"], "Diccionario y ficha no coinciden."
    roles_declarados = set(diccionario["rol_analitico"])
    assert roles_declarados, "El diccionario no declara roles analíticos."
    print(f"[OK] Diccionario coherente ({len(diccionario)} variables, "
          f"{len(roles_declarados)} roles)")

    # 5. Módulo importable
    assert (raiz / "src" / "utilidades.py").exists(), "Falta el módulo src/utilidades.py."
    assert callable(getattr(utilidades, "normalizar_columnas", None)), \
        "El módulo no expone normalizar_columnas."
    print("[OK] Módulo src/utilidades.py importable")

    print(f"\n[INFO] Conjunto declarado: {ficha.get('titulo', ficha.get('nombre', 'Sin título'))}")
    print(f"[INFO] Archivo descargado: {'sí' if ARCHIVO.exists() else 'todavía no'}")
    return True


validar_fase1(RAIZ, PROYECTO, FICHA, diccionario)

## 8. Vinculación con el mapa conceptual

La rúbrica de la Sumativa 1 evalúa por separado que el mapa conceptual de la Formativa 1
se corresponda con lo efectivamente construido. La tabla siguiente hace explícita esa
correspondencia: cada nodo del mapa, dónde está implementado y qué evidencia lo respalda.

In [ ]:
VINCULACION = [
    # (nodo del mapa conceptual, dónde se implementa, evidencia verificable)
    ("Entorno virtual",        "Sección 2 · verificar_entorno", "Ruta del intérprete y versiones"),
    ("Gestión de dependencias", "Sección 3.2 · requirements.txt", "Archivo generado desde el entorno"),
    ("Estructura del proyecto", "Sección 3 · carpetas F1–F4",     "Árbol impreso y validado"),
    ("Módulos reutilizables",   "Sección 4 · src/utilidades.py",  "Módulo importado y probado"),
    ("Fuente de datos",         "Sección 5 · FICHA y DICCIONARIO", "Ficha, cita APA 7 y roles"),
    ("Control de versiones",    "Sección 6 · consultas a Git",     "Rama, commits y estado"),
    ("Documentación científica", "Sección 9 · README y metadatos", "Archivos en docs/"),
]

vinculacion = pd.DataFrame(VINCULACION,
                           columns=["nodo_mapa", "donde_se_implementa", "evidencia"])

# Ningún nodo del mapa puede quedar sin evidencia: eso es lo que la rúbrica revisa.
assert vinculacion["evidencia"].str.strip().ne("").all(), "Hay nodos sin evidencia."
print(f"{len(vinculacion)} nodos del mapa conceptual, todos con evidencia asociada.\n")
vinculacion

## 9. Persistencia y trazabilidad

El resultado debe quedar guardado y documentado. Esta sección produce los artefactos que se
versionan en el repositorio y se citan en el informe técnico.

In [ ]:
def generar_readme(config, ficha, requisitos, ruta):
    """Escribe el README tecnico del repositorio a partir de la configuracion."""
    integrantes = "\n".join(f"- {n} (@usuario-github)" for n in config["integrantes"])
    dependencias = "\n".join(f"- {r}" for r in requisitos)

    contenido = f"""# {config['titulo']} — {config['grupo']}

{config['problematica']}

## Integrantes
{integrantes}

## Datos
- Fuente: {ficha['titulo']} ({ficha['autor']}, {ficha['plataforma']})
- Enlace: {ficha['url']}
- Estructura: {ficha['filas']} filas x {ficha['columnas']} columnas
- Ubicación esperada: `data/raw/`

## Estructura del repositorio
```
data/raw/        datos originales, sin modificar
data/processed/  datos tras limpieza y transformación (F2)
docs/            diccionario, fichas y metadatos
src/             módulos reutilizables del proyecto
F1/ F2/ F3/ F4/  cuadernos e informes de cada fase
```

## Requisitos y ejecución
Python 3.13

    python -m venv .venv
    source .venv/Scripts/activate    # Windows, Git Bash
    # source .venv/bin/activate      # macOS y Linux
    python -m pip install -r requirements.txt

Ejecutar los cuadernos en orden, F1 y luego F2. Cada cuaderno localiza la raíz del
proyecto aunque Jupyter lo abra en su propia carpeta.

### Dependencias declaradas
{dependencias}

## Convención de commits
Prefijos usados: docs, data, feat, fix.

## Decisiones técnicas
- Los datos son comunes a todas las fases (`data/`); los cuadernos se separan por fase.
- Semilla aleatoria fijada en {SEMILLA} para asegurar reproducibilidad.
- Registro breve de las decisiones relevantes y su motivo, actualizado al cerrar cada fase.
"""
    ruta.write_text(contenido, encoding="utf-8")
    return contenido


ARCHIVO_README = DIR_DOCS / "README_F1_generado.md"
contenido = generar_readme(PROYECTO, FICHA, requisitos, ARCHIVO_README)
print(f"{ARCHIVO_README} escrito ({len(contenido.splitlines())} líneas)\n")
print("\n".join(contenido.splitlines()[:14]), "\n...")